# 04 - Memory: short-term sessions vs long-term memory

**Maple Mortgage Assistant, Northwind Home Lending (fictional)**

## What you'll learn
- How an **Agent Framework** agent is built: client, `Agent(...)` with name / instructions / context providers, sessions, `run` and streaming
- **Short-term memory**: `AgentSession` keeps the turns of one conversation; it can be saved and restored
- **Long-term memory**: a Foundry **memory store** extracts durable facts (preferences) and recalls them in a *new* session
- Two ways to attach the store: in code (`FoundryMemoryProvider`) and server-side (a registered Foundry agent with a `MemorySearchPreviewTool`)
- **Scope** isolates memories per borrower; `delete_scope` implements right-to-erasure
- List/get everything back, and delete it at the end

### Where each asset lives
| Asset | Created with | Registered in Foundry? |
|---|---|---|
| `AgentSession` (short-term) | `agent.create_session()` | **No**: object in your process; you serialize it yourself |
| Agent Framework `Agent` + `FoundryMemoryProvider` | `agent_framework` | **No**: code in your process |
| Memory store `maple-borrower-memory` | `pc.beta.memory_stores.create` | **Yes**: Foundry project (Build > Memory in the portal) |
| Memories (per scope) | written by the store after each run | **Yes**: inside the store, keyed by `scope` |
| Prompt agent `maple-memory-agent` with `MemorySearchPreviewTool` | `pc.agents.create_version` | **Yes**: Foundry project (Build > Agents) |

> **[preview]** Foundry memory stores are a preview feature (`allow_preview=True`). If your tenant/region lacks it, the create cell prints the real error.

In [1]:
import os, sys, asyncio, json
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

DAY2 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(DAY2 / ".env", override=False)
sys.path.insert(0, str(DAY2))

REQUIRED = ["AZURE_AI_PROJECT_ENDPOINT", "AZURE_AI_MODEL_DEPLOYMENT_NAME", "AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
for name in REQUIRED:
    print(f"{name:42s}", "set" if os.environ.get(name) else "MISSING")   # never print values
assert all(os.environ.get(n) for n in REQUIRED), "Run scripts/write-env.sh first"

credential = DefaultAzureCredential()

import time
from agent_framework import Agent, AgentSession
from agent_framework.foundry import FoundryChatClient, FoundryMemoryProvider
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MemoryStoreDefaultDefinition, MemoryStoreDefaultOptions

ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
CHAT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
EMBED = os.environ["AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
STORE = "maple-borrower-memory"

client = FoundryChatClient(project_endpoint=ENDPOINT, model=CHAT, credential=credential)
pc = AIProjectClient(endpoint=ENDPOINT, credential=credential, allow_preview=True)   # memory stores are preview
from azure.ai.projects.aio import AIProjectClient as AsyncProjectClient
from azure.identity.aio import DefaultAzureCredential as AsyncCredential
pc_async = AsyncProjectClient(endpoint=ENDPOINT, credential=AsyncCredential(), allow_preview=True)   # the AF provider awaits its client
INSTRUCTIONS = ("You are Maple, the mortgage assistant for Northwind Home Lending (fictional). Never promise a rate or approval. "
                "Use what you remember about the borrower; if you do not remember, say so.")
oai = pc.get_openai_client()      # project OpenAI-compatible client: used to call *registered* Foundry agents
print("ready")

AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set


ready


## 1. Build an agent, run it, stream it
The pieces (all created in the setup cell and the next cell):
1. **Client**: `FoundryChatClient(project_endpoint, model, credential)` talks to the project's model deployment through the Responses API.
2. **Agent**: `Agent(client=..., name=..., instructions=..., tools=[...], context_providers=[...])`. No tools or providers yet.
3. **Session**: `agent.create_session()` holds the conversation. A brand-new session starts empty.
4. **Run**: `await agent.run(text, session=s)` returns an `AgentResponse` (`.text`, `.usage_details`).
5. **Stream**: `agent.run(text, session=s, stream=True)` returns a stream of updates (shown below).

In [2]:
agent = Agent(client=client, name="maple-short", instructions=INSTRUCTIONS)

s1 = agent.create_session()
r = await agent.run("Hi, I'm Priya. I'm a first-time buyer looking at condos around 450k.", session=s1)
print("turn 1:", r.text[:200], "\n")
r = await agent.run("What price range did I mention?", session=s1)
print("same session  :", r.text[:200], "\n")

s2 = agent.create_session()
r = await agent.run("What price range did I mention?", session=s2)
print("new session   :", r.text[:200])

turn 1: Hi Priya! It's great to hear you're looking at condos around $450,000. As a first-time buyer, there are some excellent options and programs that might be available to you. To assist you better, could  



same session  : You mentioned that you're looking at condos around $450,000. 



new session   : I don’t recall the price range you mentioned. Could you please remind me?


### Stream a reply
Same agent, same session `s1`: tokens arrive as `AgentResponseUpdate`s; the stream's final response has the full text.

In [3]:
stream = agent.run("Give me one sentence on why a first-time buyer should get pre-qualified.", session=s1, stream=True)
seen = ""
async for update in stream:
    if update.text and update.text != seen:      # the last update repeats the whole text: skip it
        print(update.text, end="", flush=True); seen += update.text
final = await stream.get_final_response()
print("\n[tokens]", final.usage_details.get("total_token_count"))

Getting

 pre

-qualified

 helps

 first

-time

... (13 outputs left out)

 the home

buying

 process.


[tokens] 230


### Save and restore a session
`to_dict()` gives JSON you can store (Cosmos, Redis, a file). Restoring it resumes the same conversation.

In [4]:
saved = json.dumps(s1.to_dict())
print("serialized session:", len(saved), "bytes")
restored = AgentSession.from_dict(json.loads(saved))
r = await agent.run("Remind me: what was my name and price range?", session=restored)
print("restored session:", r.text[:200])

serialized session: 167 bytes


restored session: Your name is Priya, and you mentioned looking at condos around $450,000.


Short-term memory is the *conversation*. It is lost when the session object is dropped and grows with every turn.
For things that must survive across conversations (preferences, goals) we need long-term memory.

## 2. Create a long-term memory store [preview]
`user_profile_enabled` extracts stable facts about the user; `chat_summary_enabled` keeps conversation summaries.

In [5]:
definition = MemoryStoreDefaultDefinition(
    chat_model=CHAT, embedding_model=EMBED,
    options=MemoryStoreDefaultOptions(user_profile_enabled=True, chat_summary_enabled=True),
)
try:
    pc.beta.memory_stores.get(STORE)
    print("store already exists:", STORE)
except Exception:
    store = pc.beta.memory_stores.create(name=STORE, definition=definition,
                                         description="Borrower preferences for Maple (fictional data only)")
    print("created store:", store.name)

store already exists: maple-borrower-memory


In [6]:
# Read the store back: get one, list all
got = pc.beta.memory_stores.get(STORE)
print("get :", got.name, "| chat model:", got.definition.chat_model, "| embedding:", got.definition.embedding_model)
print("opts:", got.definition.options.as_dict())
print("list:", [m.name for m in pc.beta.memory_stores.list()])

get : maple-borrower-memory | chat model: gpt-4.1-mini | embedding: text-embedding-3-small
opts: {'user_profile_enabled': True, 'chat_summary_enabled': True, 'default_ttl_seconds': 0}


list: ['maple-borrower-memory']


## 3. Agent with `FoundryMemoryProvider`
`scope` is the isolation key. Use an **opaque customer id** (never name, SIN/SSN, email).
`update_delay=0` writes memories right after each run (default is 300 s, which batches updates; fine for production, slow for a demo).

In [7]:
def maple_with_memory(scope: str) -> Agent:
    provider = FoundryMemoryProvider(project_client=pc_async, memory_store_name=STORE, scope=scope, update_delay=0)
    return Agent(client=client, name="maple-memory", instructions=INSTRUCTIONS, context_providers=[provider])

PRIYA = "cust-demo-001"     # opaque id, not a name
agent_p = maple_with_memory(PRIYA)

# Session A: Priya states preferences
sess_a = agent_p.create_session()
r = await agent_p.run(
    "I prefer a 5 year fixed term, I want bi-weekly payments, and I would rather not use a variable rate. "
    "I'm a first-time buyer and my budget is around 450k.", session=sess_a)
print("session A:", r.text[:300])

session A: Thanks for confirming your preferences! You’re looking for a 5-year fixed term mortgage with bi-weekly payments, avoiding variable rates, and as a first-time homebuyer with a budget around $450,000.

To help provide you with mortgage options tailored to your situation, could you please share details


Extraction runs server side. Poll the store until the facts appear (usually seconds).

In [8]:
from azure.ai.projects.models import MemorySearchOptions

def recall(scope, query, tries=1):
    for i in range(tries):
        res = pc.beta.memory_stores.search_memories(STORE, scope=scope, items=query, options=MemorySearchOptions(max_memories=5))
        mems = [m.as_dict() for m in res.memories]
        if mems or i == tries - 1:
            return mems
        time.sleep(6)

mems = recall(PRIYA, "borrower preferences", tries=12)
print(f"{len(mems)} memories for scope {PRIYA}:")
for m in mems:
    item = m.get("memory_item", m)
    print(" -", item.get("kind", ""), "|", item.get("content", item))

5 memories for scope cust-demo-001:
 - chat_summary | The user prefers a 5-year fixed term mortgage with bi-weekly payments and does not want a variable interest rate. The user is a first-time home buyer with a budget around $450,000. The assistant acknowledged the preferences and proposed exploring fixed-rate mortgage options tailored to the user's down payment, credit score, employment status, and income. The assistant requested additional details including the down payment amount or percentage, credit score information, and employment and income details to provide personalized mortgage information.
 - user_profile | User prefers bi-weekly mortgage payments.
 - chat_summary | The user previously indicated a preference for a 5-year fixed mortgage term with bi-weekly payment frequency and set a home purchase budget of around $450,000.
 - user_profile | User prefers not to use a variable interest rate for loans.
 - user_profile | User prefers a 5-year fixed term mortgage.


## 4. A new session weeks later
No history is passed: the provider retrieves the facts from the store.

In [9]:
sess_b = agent_p.create_session()          # brand new session, empty history
r = await agent_p.run("I'm back. What term and payment frequency did I prefer, and what was my budget?", session=sess_b)
print("session B:", r.text[:400])

session B: Welcome back! You preferred a 5-year fixed term mortgage with bi-weekly mortgage payments. Your home purchase budget is around $450,000. How can I assist you further today?


## 5. Isolation between borrowers
A second customer asks the same question under a different scope.

In [10]:
OTHER = "cust-demo-002"
agent_o = maple_with_memory(OTHER)
r = await agent_o.run("What term and payment frequency did I prefer?", session=agent_o.create_session())
print("other borrower:", r.text[:300])
print("memories in other scope:", len(recall(OTHER, "borrower preferences")))

other borrower: I don’t have a record of your preferred term or payment frequency. Could you please provide those details or let me know if you'd like me to help you explore options?


memories in other scope: 0


## 6. Right to erasure
`delete_scope` removes everything stored for one borrower. Verify with a search afterwards.

In [11]:
res = pc.beta.memory_stores.delete_scope(STORE, scope=PRIYA)
print("delete_scope ->", res.as_dict() if hasattr(res, "as_dict") else res)
print("memories left for", PRIYA, ":", len(recall(PRIYA, "borrower preferences")))
r = await agent_p.run("What term did I prefer?", session=agent_p.create_session())
print("after erasure :", r.text[:300])

delete_scope -> {'object': 'memory_store.scope.deleted', 'name': 'maple-borrower-memory', 'scope': 'cust-demo-001', 'deleted': True}


memories left for cust-demo-001 : 0


after erasure : I don’t recall your preferred loan term at the moment. Could you please remind me?


## 7. Attach the store server-side: a registered Foundry agent [preview]
Section 3 attached memory **in code**. Alternatively register a **prompt agent** whose tool is `MemorySearchPreviewTool(memory_store_name, scope, update_delay)`; the Foundry service then searches and updates the store for every call, and any client can call the agent by name.
`scope` is fixed in the agent definition here (one agent version per tenant/borrower segment); the portal and docs also describe a per-caller scope template.

In [12]:
from azure.ai.projects.models import PromptAgentDefinition, MemorySearchPreviewTool

SRV_AGENT, SRV_SCOPE = "maple-memory-agent", "cust-demo-003"
av = pc.agents.create_version(
    agent_name=SRV_AGENT,
    definition=PromptAgentDefinition(model=CHAT, instructions=INSTRUCTIONS, tools=[
        MemorySearchPreviewTool(memory_store_name=STORE, scope=SRV_SCOPE, update_delay=0)]))
print("registered agent:", av.name, "version", av.version, "| tools:", [t.type for t in av.definition.tools])

def call_registered(text):
    r = oai.responses.create(input=text, extra_body={"agent_reference": {"name": SRV_AGENT, "type": "agent_reference"}})
    print("tool items:", [i.type for i in r.output if i.type != "message"])
    return r.output_text

print("A1:", call_registered("I prefer a 10 year fixed term and monthly payments. I am buying a townhouse around 520k."))

registered agent: maple-memory-agent version 1 | tools: ['memory_search_preview']


tool items: ['memory_command_preview_call', 'memory_command_preview_call_output', 'memory_command_preview_call', 'memory_command_preview_call_output', 'memory_search_call']
A1: Thanks for sharing that. Since you prefer a 10-year fixed term with monthly payments and are buying a townhouse around $520,000, I can help guide you through what to expect and the next steps in the mortgage process. Do you have any questions about down payment options, loan types, or what documents you might need?


A *new* call (new conversation, no history passed). The memory tool retrieves the facts server-side:

In [13]:
recalled = None
for attempt in range(8):                 # extraction is asynchronous; poll briefly
    mems = recall(SRV_SCOPE, "borrower preferences")
    if mems: break
    time.sleep(6)
print(f"{len(mems)} memories stored for scope {SRV_SCOPE}")
print("A2:", call_registered("What term, payment frequency and budget did I tell you about?"))

1 memories stored for scope cust-demo-003


tool items: ['memory_search_call']
A2: You mentioned that you prefer a 10-year fixed mortgage term with monthly payments. You are looking to buy a townhouse priced around $520,000.


## 8. List / get everything back; portal locations
In the **Foundry portal** (project `maple-mortgage`): **Build > Memory** lists the store `maple-borrower-memory` (open it to browse memories per scope); **Build > Agents** lists `maple-memory-agent`.

In [14]:
print("memory stores :", [m.name for m in pc.beta.memory_stores.list()])
print("agents        :", sorted(a.name for a in pc.agents.list()))
print("agent versions:", [v.version for v in pc.agents.list_versions(SRV_AGENT)])
print("scope", SRV_SCOPE, "->", len(recall(SRV_SCOPE, "borrower preferences")), "memories")

memory stores : ['maple-borrower-memory']


agents        : ['maple-hosted', 'maple-learn-mcp', 'maple-memory-agent', 'maple-prompt']


agent versions: ['1']


scope cust-demo-003 -> 2 memories


## 9. Clean up
Order matters: erase memories (scope), then the agent that references the store, then the store.
`delete_scope` is the right-to-erasure call; deleting the store removes every scope in it.

In [15]:
for scope in (SRV_SCOPE, OTHER):          # PRIYA was already erased in section 6
    print("delete_scope", scope, "->", pc.beta.memory_stores.delete_scope(STORE, scope=scope).as_dict().get("deleted", "ok"))
pc.agents.delete(SRV_AGENT)
print("deleted agent:", SRV_AGENT)
print("delete store ->", pc.beta.memory_stores.delete(STORE).as_dict())
print("memory stores left:", [m.name for m in pc.beta.memory_stores.list()])

delete_scope cust-demo-003 -> True


delete_scope cust-demo-002 -> True


deleted agent: maple-memory-agent


delete store -> {'object': 'memory_store.deleted', 'name': 'maple-borrower-memory', 'deleted': True}
memory stores left: []


## FSI notes: PII and retention
| Topic | Guidance |
|---|---|
| What to store | Preferences and goals (term, payment frequency, budget band). **Not** SIN/SSN, full account numbers, health or credit-bureau data. |
| Key | Opaque customer id as `scope`; map to the person in your CRM, not in the memory store. |
| Redaction | Mask PII before it reaches the model/memory (guardrail or pre-processing); the store keeps what the model saw. |
| Retention | Define a TTL per policy (e.g. delete scope at account closure or after N months of inactivity) and run `delete_scope` from a scheduled job. |
| Erasure | `delete_scope` covers right-to-erasure requests; log the request id, not the content. |
| Audit | Memory is *derived* data and can be wrong; do not use it for credit decisions. Keep decisions based on verified application data. |
| Access | Entra ID only; the caller needs Azure AI User on the project. Keep the store in the same region/residency boundary as the project. |

## Key takeaways
- Session = this conversation; memory store = facts across conversations.
- `scope` is your tenant/borrower boundary: test isolation explicitly.
- Plan deletion (retention + erasure) before you plan recall.
- Memory stores are preview: pin versions and keep a fallback (your own store of preferences).

**Next up:** [05 - Skills and registry](05-skills-and-registry.ipynb)